# Multi-Asset CTA Strategy V2 — Transition Strategy

## 06 — Economic Validation and Position Construction (Event-Level)

Books 01–05 established the data architecture, transition-event definition, candidate-time feature engine, frozen Book 04 probability model, and the rejection of HMM complexity. Book 06 asks the stricter economic question:

\[
\boxed{\text{Does the frozen Book 04 transition probability contain economically exploitable information?}}
\]

Classification quality is not trading alpha. This book therefore freezes `RF_unweighted_C_plus_SC_no_MACD` and tests whether higher candidate-time probability maps into superior **candidate-direction forward returns** and sensible anticipatory exposure.

### Governing principles

- Freeze the Book 04 signal; no retraining or feature discovery.
- Use exact saved annual OOS probabilities.
- Evaluate bear→bull candidates long and bull→bear candidates short.
- Prefer Book 01 prototype P&L prices; use signal prices only as an explicitly labelled fallback.
- Treat this as an event study, not yet a capital-constrained portfolio backtest.
- Fix probability buckets and sizing maps ex ante; do not threshold-mine.
- Preserve direction, asset-class, year, Bitcoin-exclusion and P&L-proxy robustness.
- Use market-local observations for forward horizons.

### Primary hypotheses

1. Higher frozen transition probability produces better candidate-direction forward returns.
2. The top probability quintile economically outperforms the unconditional candidate set.
3. High-probability failed transitions do not erase genuine-transition gains.
4. Separation appears early enough to justify anticipatory positioning.
5. Simple monotonic probability-to-exposure maps preserve/improve event-level economics.
6. Results have breadth across years, directions and asset classes.

In [ ]:
# =========================================================
# 1) INSTALLS, IMPORTS, DRIVE, PATHS, CONFIG
# =========================================================
!pip -q install pyarrow
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
warnings.filterwarnings('ignore')

PROJECT=Path('/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2')
V201=PROJECT/'v2.01'; V203=PROJECT/'v2.03'; V204=PROJECT/'v2.04'; V206=PROJECT/'v2.06'
CONFIG_DIR=V206/'config'; DATA_DIR=V206/'data'; RESULTS_DIR=V206/'results'
for p in [CONFIG_DIR,DATA_DIR,RESULTS_DIR]: p.mkdir(parents=True,exist_ok=True)
PATHS={
 'book04_predictions':V204/'data'/'v2_04_unweighted_predictions.parquet',
 'candidate_features_parquet':V203/'data'/'v2_03_candidate_features.parquet',
 'candidate_features_csv':V203/'results'/'v2_03_candidate_features.csv',
 'pnl_prices':V201/'data'/'processed'/'v2_01_pnl_prices_prototype.parquet',
 'signal_prices':V201/'data'/'processed'/'v2_01_signal_prices.parquet'}
CONFIG={
 'book':'V2.06','frozen_model':'RF_unweighted_C_plus_SC_no_MACD','research_end':'2025-12-31',
 'first_oos_year':2008,'forward_horizons':[5,10,21,42,63,126],
 'primary_quantiles':5,'secondary_deciles':10,'primary_horizon':63,
 'cost_bps_grid':[0,5,10,25],'bootstrap_draws':2000,'bootstrap_seed':42}
print('Book 06 output:',V206); print(json.dumps(CONFIG,indent=2))

Mounted at /content/drive
Book 06 output: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.06
{
  "book": "V2.06",
  "frozen_model": "RF_unweighted_C_plus_SC_no_MACD",
  "research_end": "2025-12-31",
  "first_oos_year": 2008,
  "forward_horizons": [
    5,
    10,
    21,
    42,
    63,
    126
  ],
  "primary_quantiles": 5,
  "secondary_deciles": 10,
  "primary_horizon": 63,
  "cost_bps_grid": [
    0,
    5,
    10,
    25
  ],
  "bootstrap_draws": 2000,
  "bootstrap_seed": 42
}


In [ ]:
# =========================================================
# 2) LOAD BOOK 03 CANDIDATES + EXACT FROZEN BOOK 04 OOS PROBABILITIES
# =========================================================
if PATHS['candidate_features_parquet'].exists():
    candidates=pd.read_parquet(PATHS['candidate_features_parquet']); candidate_source=PATHS['candidate_features_parquet']
elif PATHS['candidate_features_csv'].exists():
    candidates=pd.read_csv(PATHS['candidate_features_csv']); candidate_source=PATHS['candidate_features_csv']
else: raise FileNotFoundError('Book 03 candidate features not found.')
candidates['candidate_date']=pd.to_datetime(candidates['candidate_date'])
candidates['target']=(candidates['label'].astype(str).str.lower()=='genuine').astype(int)
candidates['test_year']=candidates['candidate_date'].dt.year

pred=pd.read_parquet(PATHS['book04_predictions']); pred['candidate_date']=pd.to_datetime(pred['candidate_date'])
frozen=pred[pred['model'].astype(str)==CONFIG['frozen_model']].copy()
prob_col=next((c for c in ['prob_genuine','predicted_probability','probability','prediction'] if c in frozen.columns),None)
if prob_col is None: raise ValueError('Book 04 probability column not found.')
frozen=frozen.rename(columns={prob_col:'prob_genuine'})
merge_keys=['candidate_date','market']
for x in ['candidate_direction','category']:
    if x in candidates.columns and x in frozen.columns: merge_keys.append(x)
base=candidates.merge(frozen[merge_keys+['prob_genuine']].drop_duplicates(merge_keys),on=merge_keys,how='inner',validate='one_to_one')
base=base[(base.test_year>=CONFIG['first_oos_year'])&(base.candidate_date<=pd.Timestamp(CONFIG['research_end']))].copy()
print('Frozen OOS events:',len(base)); print('Genuine rate:',base.target.mean()); print('Mean probability:',base.prob_genuine.mean())

Frozen OOS events: 2687
Genuine rate: 0.32526981764049123
Mean probability: 0.3241192405403636


## Return-series policy

Book 06 preserves the Book 01 distinction `Signal Series != P&L Series`. Prototype P&L prices are preferred where a usable market series exists. Otherwise the signal-price series is used only as a labelled event-study fallback. This broad test does not yet solve futures roll/collateral economics, FX carry, or institutional execution costs; those belong in the portfolio implementation stage.

In [ ]:
# =========================================================
# 3) NORMALISE PRICE PANELS + MATCH MARKETS
# =========================================================
def normalise_price_panel(df):
    x=df.copy(); lower={str(c).lower():c for c in x.columns}
    dc=next((lower[k] for k in ['date','datetime','timestamp'] if k in lower),None)
    mc=next((lower[k] for k in ['market','asset','name'] if k in lower),None)
    pc=next((lower[k] for k in ['price','close','signal_price','pnl_price','value'] if k in lower),None)
    if dc is not None and mc is not None and pc is not None:
        x[dc]=pd.to_datetime(x[dc]); return x.pivot_table(index=dc,columns=mc,values=pc,aggfunc='last').sort_index()
    if dc is not None: x[dc]=pd.to_datetime(x[dc]); x=x.set_index(dc)
    if not isinstance(x.index,pd.DatetimeIndex): x.index=pd.to_datetime(x.index)
    return x.apply(pd.to_numeric,errors='coerce').sort_index()

signal_prices=normalise_price_panel(pd.read_parquet(PATHS['signal_prices'])).loc[:pd.Timestamp(CONFIG['research_end'])]
pnl_prices=normalise_price_panel(pd.read_parquet(PATHS['pnl_prices'])).loc[:pd.Timestamp(CONFIG['research_end'])] if PATHS['pnl_prices'].exists() else pd.DataFrame()

def canon(s):
    return str(s).strip().lower().replace('&','and').replace('/','').replace('-','').replace('_','').replace(' ','').replace('.','').replace('^','')
def match_market(m,cols):
    cols=list(cols)
    if m in cols:return m
    lookup={canon(c):c for c in cols}; cm=canon(m)
    if cm in lookup:return lookup[cm]
    hits=[c for c in cols if cm in canon(c) or canon(c) in cm]
    return hits[0] if len(hits)==1 else None

market_return_map={}
for m in sorted(base.market.astype(str).unique()):
    pm=match_market(m,pnl_prices.columns); sm=match_market(m,signal_prices.columns)
    market_return_map[m]={'source':'PNL_PROTOTYPE','column':pm} if pm is not None else ({'source':'SIGNAL_FALLBACK','column':sm} if sm is not None else {'source':'UNMATCHED','column':None})
return_map_df=pd.DataFrame([{'market':m,**v} for m,v in market_return_map.items()])
display(return_map_df.source.value_counts().rename('markets'))

,markets
source,
PNL_PROTOTYPE,52
SIGNAL_FALLBACK,1


In [ ]:
# =========================================================
# 4) LOCAL-CALENDAR CANDIDATE-DIRECTION FORWARD RETURNS
# =========================================================
def direction_sign(x):
    if pd.isna(x):return np.nan
    if isinstance(x,str):
        s=x.upper()
        if 'BEAR' in s and 'BULL' in s and s.index('BEAR')<s.index('BULL'):return 1.0
        if 'BULL' in s and 'BEAR' in s and s.index('BULL')<s.index('BEAR'):return -1.0
        try:return 1.0 if float(x)>0 else -1.0
        except:return np.nan
    return 1.0 if float(x)>0 else -1.0

def get_series(m):
    z=market_return_map[str(m)]
    if z['source']=='PNL_PROTOTYPE':s=pnl_prices[z['column']]
    elif z['source']=='SIGNAL_FALLBACK':s=signal_prices[z['column']]
    else:return None,z['source']
    s=s.dropna().astype(float); s=s[np.isfinite(s)&(s>0)].sort_index(); return s,z['source']

rows=[]
for _,r in base.iterrows():
    s,source=get_series(r.market)
    if s is None or len(s)==0:continue
    pos=int(s.index.searchsorted(pd.Timestamp(r.candidate_date),side='left'))
    if pos>=len(s):continue
    d=direction_sign(r.candidate_direction)
    if not np.isfinite(d):continue
    o=r.to_dict(); o.update({'return_source':source,'entry_date':s.index[pos],'entry_price':float(s.iloc[pos]),'direction_sign':d})
    for h in CONFIG['forward_horizons']:
        j=pos+h
        if j<len(s):
            rr=float(s.iloc[j]/s.iloc[pos]-1); o[f'raw_fwd_{h}']=rr; o[f'dir_fwd_{h}']=d*rr; o[f'exit_date_{h}']=s.index[j]
        else:o[f'raw_fwd_{h}']=np.nan;o[f'dir_fwd_{h}']=np.nan;o[f'exit_date_{h}']=pd.NaT
    rows.append(o)
events=pd.DataFrame(rows)
print('Events with returns:',len(events),'coverage',len(events)/len(base)); display(events.return_source.value_counts())

Events with returns: 2687 coverage 1.0


,count
return_source,
PNL_PROTOTYPE,2625
SIGNAL_FALLBACK,62


In [ ]:
# =========================================================
# 5) FIXED YEARLY PROBABILITY QUINTILES / DECILES
# =========================================================
def add_bucket(df,n,name):
    out=df.copy(); out[name]=np.nan
    for y,g in out.groupby('test_year'):
        if len(g)<n:continue
        ranks=g.prob_genuine.rank(method='first',pct=True)
        out.loc[g.index,name]=np.ceil(ranks*n).clip(1,n).astype(int)
    out[name]=out[name].astype('Int64'); return out
events=add_bucket(events,5,'prob_quintile'); events=add_bucket(events,10,'prob_decile')
events['direction_name']=np.where(events.direction_sign>0,'BEAR_TO_BULL','BULL_TO_BEAR')
print(events[['prob_genuine','prob_quintile','prob_decile']].describe())

       prob_genuine  prob_quintile  prob_decile
count   2687.000000         2687.0       2687.0
mean       0.324119       3.011165     5.524377
std        0.156189       1.413643     2.872712
min        0.019345            1.0          1.0
25%        0.200979            2.0          3.0
50%        0.317591            3.0          6.0
75%        0.431034            4.0          8.0
max        0.776883            5.0         10.0


## Primary economic test — probability → payoff monotonicity

For each fixed probability bucket and horizon, Book 06 reports candidate-direction mean/median return, positive-return rate and genuine-transition rate. The desired result is monotonic or near-monotonic economic improvement as probability rises, not merely one lucky bucket.

In [ ]:
# =========================================================
# 6) QUINTILE / DECILE RETURN CURVES + MONOTONICITY
# =========================================================
def bucket_table(df,bcol):
    rows=[]
    for b,g in df.dropna(subset=[bcol]).groupby(bcol):
        for h in CONFIG['forward_horizons']:
            c=f'dir_fwd_{h}'; x=g[c].dropna()
            if len(x):rows.append({'bucket_type':bcol,'bucket':int(b),'horizon':h,'events':len(x),'genuine_rate':g.loc[x.index,'target'].mean(),'mean_dir_return':x.mean(),'median_dir_return':x.median(),'positive_return_rate':(x>0).mean(),'return_std':x.std(),'mean_probability':g.loc[x.index,'prob_genuine'].mean()})
    return pd.DataFrame(rows)
quintile_returns=bucket_table(events,'prob_quintile'); decile_returns=bucket_table(events,'prob_decile')

def spear(x,y):
    a=pd.Series(x).rank().to_numpy(float);b=pd.Series(y).rank().to_numpy(float)
    return float(np.corrcoef(a,b)[0,1]) if len(a)>2 and np.std(a)>0 and np.std(b)>0 else np.nan
mr=[]
for h in CONFIG['forward_horizons']:
    q=quintile_returns[quintile_returns.horizon==h].sort_values('bucket'); c=f'dir_fwd_{h}'; u=events.dropna(subset=[c,'prob_quintile']); top=u[u.prob_quintile==5]
    mr.append({'horizon':h,'spearman_bucket_vs_mean_return':spear(q.bucket,q.mean_dir_return) if len(q)==5 else np.nan,'spearman_bucket_vs_median_return':spear(q.bucket,q.median_dir_return) if len(q)==5 else np.nan,'all_events_mean_return':u[c].mean(),'top_quintile_mean_return':top[c].mean(),'top_minus_all_mean_return':top[c].mean()-u[c].mean(),'all_events_positive_rate':(u[c]>0).mean(),'top_quintile_positive_rate':(top[c]>0).mean(),'top_minus_all_positive_rate':(top[c]>0).mean()-(u[c]>0).mean()})
monotonicity=pd.DataFrame(mr)
display(quintile_returns[quintile_returns.horizon==CONFIG['primary_horizon']]); display(monotonicity)

,bucket_type,bucket,horizon,events,genuine_rate,mean_dir_return,median_dir_return,positive_return_rate,return_std,mean_probability
4,prob_quintile,1,63,523,0.103250,-0.004821,-0.006524,0.485660,0.133508,0.120049
10,prob_quintile,2,63,528,0.187500,-0.011605,-0.001105,0.492424,0.114918,0.225383
16,prob_quintile,3,63,533,0.262664,-0.002678,-0.001450,0.478424,0.109759,0.315760
22,prob_quintile,4,63,535,0.437383,-0.000597,0.000355,0.512150,0.193253,0.406342
28,prob_quintile,5,63,540,0.637037,0.005937,0.001723,0.509259,0.120094,0.546816


,horizon,spearman_bucket_vs_mean_return,spearman_bucket_vs_median_return,all_events_mean_return,top_quintile_mean_return,top_minus_all_mean_return,all_events_positive_rate,top_quintile_positive_rate,top_minus_all_positive_rate
0,5,-0.1,0.5,-0.000799,-0.000923,-0.000124,0.448957,0.452206,0.003249
1,10,0.9,0.3,-0.003464,-0.000455,0.003009,0.453393,0.452206,-0.001187
2,21,1.0,0.5,-0.003230,0.003220,0.006449,0.463952,0.474265,0.010313
3,42,0.6,0.9,-0.001574,0.005139,0.006713,0.477494,0.479705,0.002210
4,63,0.9,0.9,-0.002704,0.005937,0.008641,0.495675,0.509259,0.013584
5,126,0.9,0.7,-0.002603,0.008981,0.011583,0.489935,0.504708,0.014773


In [ ]:
# =========================================================
# 7) GENUINE/FAILED PAYOFF DECOMPOSITION + ROBUSTNESS
# =========================================================
pr=[]
for h in CONFIG['forward_horizons']:
    c=f'dir_fwd_{h}'
    for scope,g0 in [('ALL',events),('TOP_QUINTILE',events[events.prob_quintile==5])]:
        for t,g in g0.groupby('target'):
            x=g[c].dropna()
            if len(x):pr.append({'scope':scope,'horizon':h,'target':int(t),'label':'GENUINE' if t==1 else 'FAILED','events':len(x),'mean_dir_return':x.mean(),'median_dir_return':x.median(),'positive_return_rate':(x>0).mean(),'mean_probability':g.loc[x.index,'prob_genuine'].mean()})
payoff_decomposition=pd.DataFrame(pr)

def subgroup(df,cols):
    h=CONFIG['primary_horizon'];c=f'dir_fwd_{h}';rows=[]; grouper=cols[0] if len(cols)==1 else cols
    for keys,g in df.groupby(grouper):
        if not isinstance(keys,tuple):keys=(keys,)
        u=g.dropna(subset=[c,'prob_quintile']);top=u[u.prob_quintile==5]
        if not len(u) or not len(top):continue
        r=dict(zip(cols,keys));r.update({'events':len(u),'top_events':len(top),'all_mean_return':u[c].mean(),'top_mean_return':top[c].mean(),'top_minus_all':top[c].mean()-u[c].mean(),'all_positive_rate':(u[c]>0).mean(),'top_positive_rate':(top[c]>0).mean(),'all_genuine_rate':u.target.mean(),'top_genuine_rate':top.target.mean()});rows.append(r)
    return pd.DataFrame(rows)
metrics_by_direction=subgroup(events,['direction_name']);metrics_by_asset_class=subgroup(events,['category']);metrics_by_year=subgroup(events,['test_year'])
traditional=events[events.category.astype(str).str.upper()!='DIGITAL_ASSETS'].copy()
metrics_traditional=subgroup(pd.concat([events.assign(sample='ALL_ASSETS'),traditional.assign(sample='TRADITIONAL_ONLY')]),['sample'])
pnl_only=events[events.return_source=='PNL_PROTOTYPE'].copy(); pnl_robustness=[]
for h in CONFIG['forward_horizons']:
    c=f'dir_fwd_{h}'
    for sn,g in [('ALL_RETURN_SOURCES',events),('PNL_PROTOTYPE_ONLY',pnl_only)]:
        u=g.dropna(subset=[c,'prob_quintile']);top=u[u.prob_quintile==5]
        if len(top):pnl_robustness.append({'sample':sn,'horizon':h,'events':len(u),'top_events':len(top),'all_mean_return':u[c].mean(),'top_mean_return':top[c].mean(),'top_minus_all':top[c].mean()-u[c].mean(),'top_positive_rate':(top[c]>0).mean(),'top_genuine_rate':top.target.mean()})
pnl_robustness=pd.DataFrame(pnl_robustness)
display(metrics_by_direction);display(metrics_by_asset_class);display(metrics_traditional)

,direction_name,events,top_events,all_mean_return,top_mean_return,top_minus_all,all_positive_rate,top_positive_rate,all_genuine_rate,top_genuine_rate
0,BEAR_TO_BULL,1191,300,0.004385,0.013810,0.009426,0.544081,0.570000,0.353484,0.673333
1,BULL_TO_BEAR,1468,240,-0.008455,-0.003906,0.004550,0.456403,0.433333,0.306540,0.591667


,category,events,top_events,all_mean_return,top_mean_return,top_minus_all,all_positive_rate,top_positive_rate,all_genuine_rate,top_genuine_rate
0,BONDS_RATES,325,64,-0.002723,-0.006557,-0.003833,0.483077,0.484375,0.289231,0.546875
1,COMMODITIES,875,187,0.006163,0.002916,-0.003248,0.491429,0.470588,0.348571,0.679144
2,DIGITAL_ASSETS,31,8,0.048702,0.168728,0.120025,0.677419,0.875000,0.451613,1.000000
3,FX,578,137,0.002942,0.002529,-0.000413,0.531142,0.510949,0.366782,0.635036
4,INDICES,850,144,-0.017539,0.009610,0.027149,0.474118,0.548611,0.289412,0.604167


,sample,events,top_events,all_mean_return,top_mean_return,top_minus_all,all_positive_rate,top_positive_rate,all_genuine_rate,top_genuine_rate
0,ALL_ASSETS,2659,540,-0.002704,0.005937,0.008641,0.495675,0.509259,0.327567,0.637037
1,TRADITIONAL_ONLY,2628,532,-0.003310,0.003489,0.006799,0.493531,0.503759,0.326104,0.631579


## Fixed anticipatory sizing maps

Book 06 does not optimize thresholds. It tests four simple mappings capped at **50% anticipatory exposure**: top-quintile only, a 0/0/10/25/50% quintile staircase, linear probability sizing, and excess-probability sizing above an expanding prior base rate. These are event-level research mappings, not final portfolio allocations.

In [ ]:
# =========================================================
# 8) CHRONOLOGICAL SIZING MAPS + SIMPLE COST SENSITIVITY
# =========================================================
events=events.sort_values(['test_year','candidate_date','market']).copy(); events['prior_base_rate']=np.nan
for y in sorted(events.test_year.unique()):
    prior=base[base.test_year<y]
    if len(prior):events.loc[events.test_year==y,'prior_base_rate']=prior.target.mean()

def exposure(r,name):
    p=float(r.prob_genuine);q=r.prob_quintile;b=r.prior_base_rate
    if name=='TOP_QUINTILE_50':return .5 if q==5 else 0.
    if name=='QUINTILE_STAIRCASE':return [0,0,.10,.25,.50][int(q)-1] if pd.notna(q) else np.nan
    if name=='PROBABILITY_LINEAR_50':return .5*np.clip(p,0,1)
    if name=='EXCESS_PROBABILITY_50':return np.nan if not np.isfinite(b) else (0. if p<=b else .5*np.clip((p-b)/(1-b),0,1))
MAPS=['TOP_QUINTILE_50','QUINTILE_STAIRCASE','PROBABILITY_LINEAR_50','EXCESS_PROBABILITY_50']
for m in MAPS:events['exposure_'+m]=events.apply(lambda r:exposure(r,m),axis=1)

sr=[]
for m in MAPS:
    e='exposure_'+m
    for h in CONFIG['forward_horizons']:
        r=f'dir_fwd_{h}';g=events.dropna(subset=[e,r]);gross=g[e]*g[r]
        for bps in CONFIG['cost_bps_grid']:
            net=gross-g[e].abs()*(bps/10000);active=g[e]>0
            sr.append({'sizing_map':m,'horizon':h,'cost_bps':bps,'events':len(g),'active_events':int(active.sum()),'mean_exposure':g[e].mean(),'mean_gross_event_return':gross.mean(),'mean_net_event_return':net.mean(),'median_net_event_return':net.median(),'positive_net_event_rate':(net>0).mean(),'active_mean_net_return':net[active].mean() if active.any() else np.nan,'active_positive_rate':(net[active]>0).mean() if active.any() else np.nan})
sizing_economics=pd.DataFrame(sr)
display(sizing_economics[(sizing_economics.horizon==63)&(sizing_economics.cost_bps==0)])

,sizing_map,horizon,cost_bps,events,active_events,mean_exposure,mean_gross_event_return,mean_net_event_return,median_net_event_return,positive_net_event_rate,active_mean_net_return,active_positive_rate
16,TOP_QUINTILE_50,63,0,2659,540,0.101542,0.000603,0.000603,0.000000,0.103422,0.002968,0.509259
40,QUINTILE_STAIRCASE,63,0,2659,1608,0.171888,0.000519,0.000519,-0.000000,0.302369,0.000858,0.500000
64,PROBABILITY_LINEAR_50,63,0,2659,2659,0.162234,-0.000146,-0.000146,-0.000022,0.495675,-0.000146,0.495675
88,EXCESS_PROBABILITY_50,63,0,2541,1161,0.044310,0.000100,0.000100,-0.000000,0.230224,0.000219,0.503876


## Dependence-aware uncertainty

Candidate events are not IID: many markets can transition in the same macro episode and events can overlap. Book 06 therefore uses a **year-cluster bootstrap** for the top-quintile minus all-candidate return spread, resampling complete OOS years rather than individual events.

In [ ]:
# =========================================================
# 9) YEAR-CLUSTER BOOTSTRAP
# =========================================================
rng=np.random.default_rng(CONFIG['bootstrap_seed'])
def boot(df,h):
    c=f'dir_fwd_{h}';d=df.dropna(subset=[c,'prob_quintile']);years=np.array(sorted(d.test_year.unique()));vals=[]
    for _ in range(CONFIG['bootstrap_draws']):
        sy=rng.choice(years,size=len(years),replace=True);b=pd.concat([d[d.test_year==y] for y in sy],ignore_index=True);top=b[b.prob_quintile==5]
        if len(top):vals.append(top[c].mean()-b[c].mean())
    vals=np.asarray(vals,float);obs=d.loc[d.prob_quintile==5,c].mean()-d[c].mean()
    return {'horizon':h,'draws':len(vals),'observed_spread':obs,'bootstrap_mean':vals.mean(),'ci_2_5':np.quantile(vals,.025),'ci_50':np.quantile(vals,.5),'ci_97_5':np.quantile(vals,.975),'prob_spread_positive':(vals>0).mean()}
bootstrap_results=pd.DataFrame([boot(events,h) for h in CONFIG['forward_horizons']]);display(bootstrap_results)

,horizon,draws,observed_spread,bootstrap_mean,ci_2_5,ci_50,ci_97_5,prob_spread_positive
0,5,2000,-0.000124,-0.000169,-0.003064,-0.000230,0.002852,0.4400
1,10,2000,0.003009,0.002931,-0.001784,0.002838,0.008318,0.8790
2,21,2000,0.006449,0.006493,0.000360,0.006519,0.012337,0.9820
3,42,2000,0.006713,0.006625,-0.001348,0.006828,0.013418,0.9535
4,63,2000,0.008641,0.008730,-0.002030,0.008572,0.019415,0.9485
5,126,2000,0.011583,0.011498,-0.004303,0.011800,0.026736,0.9240


# Completion Gate

Book 06 promotes the frozen Book 04 signal toward portfolio implementation only if:

1. forward returns improve with probability;
2. the top quintile has economically superior payoff/hit rate at useful horizons;
3. separation appears early enough for anticipatory positioning;
4. failed-transition losses do not overwhelm genuine-transition gains;
5. fixed monotonic sizing preserves/improves economics without optimization;
6. results have breadth across years, directions and asset classes;
7. excluding Bitcoin does not change the conclusion;
8. the P&L-prototype subset agrees directionally with the broad event study; and
9. year-cluster uncertainty does not show that a few macro episodes drive the result.

If these conditions pass, Book 07 should implement the actual portfolio state machine: incumbent conventional-trend exposure, anticipatory sleeve sizing, confirmation hand-off, invalidation, overlapping signals, volatility targeting, cross-market capital allocation, turnover, transaction costs and investable return construction.

In [ ]:
# =========================================================
# 10) SAVE OUTPUTS
# =========================================================
outputs={
'event_panel':DATA_DIR/'v2_06_event_economic_panel.parquet','quintile_returns':RESULTS_DIR/'v2_06_quintile_forward_returns.csv','decile_returns':RESULTS_DIR/'v2_06_decile_forward_returns.csv','monotonicity':RESULTS_DIR/'v2_06_probability_return_monotonicity.csv','payoff_decomposition':RESULTS_DIR/'v2_06_genuine_failed_payoff_decomposition.csv','metrics_by_direction':RESULTS_DIR/'v2_06_metrics_by_direction.csv','metrics_by_asset_class':RESULTS_DIR/'v2_06_metrics_by_asset_class.csv','metrics_by_year':RESULTS_DIR/'v2_06_metrics_by_year.csv','traditional_robustness':RESULTS_DIR/'v2_06_traditional_only_robustness.csv','pnl_robustness':RESULTS_DIR/'v2_06_pnl_proxy_robustness.csv','sizing_economics':RESULTS_DIR/'v2_06_sizing_economics.csv','bootstrap':RESULTS_DIR/'v2_06_year_cluster_bootstrap.csv','return_map':RESULTS_DIR/'v2_06_return_source_map.csv','config':CONFIG_DIR/'v2_06_config.json'}
events.to_parquet(outputs['event_panel'],index=False);quintile_returns.to_csv(outputs['quintile_returns'],index=False);decile_returns.to_csv(outputs['decile_returns'],index=False);monotonicity.to_csv(outputs['monotonicity'],index=False);payoff_decomposition.to_csv(outputs['payoff_decomposition'],index=False);metrics_by_direction.to_csv(outputs['metrics_by_direction'],index=False);metrics_by_asset_class.to_csv(outputs['metrics_by_asset_class'],index=False);metrics_by_year.to_csv(outputs['metrics_by_year'],index=False);metrics_traditional.to_csv(outputs['traditional_robustness'],index=False);pnl_robustness.to_csv(outputs['pnl_robustness'],index=False);sizing_economics.to_csv(outputs['sizing_economics'],index=False);bootstrap_results.to_csv(outputs['bootstrap'],index=False);return_map_df.to_csv(outputs['return_map'],index=False)
with open(outputs['config'],'w') as f:json.dump(CONFIG,f,indent=2,default=str)
for k,p in outputs.items():print(k,':',p)

event_panel : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.06/data/v2_06_event_economic_panel.parquet
quintile_returns : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.06/results/v2_06_quintile_forward_returns.csv
decile_returns : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.06/results/v2_06_decile_forward_returns.csv
monotonicity : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.06/results/v2_06_probability_return_monotonicity.csv
payoff_decomposition : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.06/results/v2_06_genuine_failed_payoff_decomposition.csv
metrics_by_direction : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy v2/v2.06/results/v2_06_metrics_by_direction.csv
metrics_by_asset_class : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strate

## Outputs to upload for analysis

### Required
1. `v2_06_probability_return_monotonicity.csv`
2. `v2_06_quintile_forward_returns.csv`
3. `v2_06_genuine_failed_payoff_decomposition.csv`
4. `v2_06_metrics_by_direction.csv`
5. `v2_06_metrics_by_asset_class.csv`
6. `v2_06_metrics_by_year.csv`
7. `v2_06_sizing_economics.csv`
8. `v2_06_year_cluster_bootstrap.csv`
9. `v2_06_pnl_proxy_robustness.csv`
10. `v2_06_return_source_map.csv`

### Preferred
11. `v2_06_decile_forward_returns.csv`
12. `v2_06_traditional_only_robustness.csv`

The large `v2_06_event_economic_panel.parquet` is useful for event-level diagnosis but is not required for the first results review.

## Research Outcome

Book 06 tested whether the frozen Book 04 transition probability contains economically exploitable information rather than merely classifying eventual conventional trend confirmation.

The central result is positive but materially more nuanced than the classification results alone suggest.

Higher predicted transition probability is associated with progressively better candidate-direction forward returns once sufficient time has elapsed after the initial transition candidate. The effect is **not immediate**: at 5 observations the highest probability quintile provides essentially no economic advantage, with a top-minus-all mean-return spread of approximately **−0.01%**. The relationship begins to emerge by 10 observations and becomes substantially clearer from approximately 21 observations onward.

Probability-quintile ordering of mean returns is perfectly monotonic at 21 market observations (Spearman \( \rho=1.0 \)) and remains strongly monotonic at 63 and 126 observations (\( \rho=0.9 \)).

Relative to the unconditional candidate population, the highest probability quintile improves mean candidate-direction returns by approximately:

- **−0.01%** after 5 observations;
- **+0.30%** after 10 observations;
- **+0.64%** after 21 observations;
- **+0.67%** after 42 observations;
- **+0.86%** after 63 observations;
- **+1.16%** after 126 observations.

The 21-observation result provides the strongest dependence-aware statistical evidence. Its top-quintile return spread is approximately **+0.645%**, with a year-cluster bootstrap 95% interval of approximately **+0.036% to +1.234%** and a **98.2% bootstrap probability that the spread is positive**.

This timing pattern is itself economically informative. The Book 04 probability signal appears better suited to identifying an **emerging transition process** than to pinpointing the exact reversal date. The evidence therefore argues against automatically establishing maximum anticipatory exposure immediately when a transition candidate first appears. Book 07 should explicitly test delayed, progressive and probability-dependent accumulation.

The economic distinction between genuine and failed transitions is substantial. At 63 observations, genuine candidates subsequently return approximately **+5.40%** in the predicted transition direction, while failed candidates return approximately **−3.03%**. Within the highest probability quintile, genuine transitions return approximately **+4.55%**, whereas high-confidence failures lose approximately **−6.35%**.

This establishes an important implementation constraint: increasing transition probability improves the probability of identifying a genuine transition, but false positives remain economically costly. **Invalidation and exposure control are therefore integral components of the transition strategy rather than optional risk overlays.** A successful implementation must capture the improved genuine-transition frequency without allowing high-confidence failures to dominate the payoff distribution.

The results are also directionally asymmetric. At the 63-observation horizon, top-quintile bear→bull candidates produce an average candidate-direction return of approximately **+1.38%**, compared with **+0.44%** across all bear→bull candidates, an improvement of approximately **+0.94 percentage points**.

By contrast, top-quintile bull→bear candidates remain negative at approximately **−0.39%**, despite improving by approximately **+0.46 percentage points** relative to the approximately **−0.85%** return of the full bull→bear candidate population.

Consequently, Book 04's ability to identify deterioration of established bull trends does not automatically translate into profitable anticipatory shorting. The production strategy should therefore **not impose symmetric long and short transition exposure**. Bear→bull and bull→bear transitions should be treated as economically distinct implementation problems even though they originate from the same probabilistic classification framework.

Cross-asset results are heterogeneous. Equity indices show particularly strong economic ranking at the 63-observation horizon, with the top probability quintile improving mean candidate-direction return by approximately **+2.71 percentage points** relative to all index candidates. Commodities, FX and bonds/rates show weaker economic separation at this horizon.

Bitcoin produces unusually strong returns but has a very small event sample and is not permitted to determine the architecture. Excluding digital assets, the top-quintile return advantage remains positive at approximately **+0.68 percentage points**, confirming that the broad economic result is not dependent on Bitcoin.

Book 06 therefore supports the existence of **economic transition information** in the frozen Book 04 probability signal, while rejecting the interpretation that classification quality alone justifies indiscriminate anticipatory trading.

The fixed sizing experiments should consequently be interpreted as evidence about the feasibility of translating probability into exposure rather than as optimization of a finished trading rule. The appropriate production problem is not simply whether to trade a transition candidate, but:

\[
\text{when to begin}
\quad+\quad
\text{how quickly to accumulate}
\quad+\quad
\text{how much risk to allocate}
\quad+\quad
\text{when to invalidate}
\quad+\quad
\text{when to hand off to conventional trend}.
\]

The evidence supports progression to portfolio construction subject to five requirements:

1. **transition exposure must increase conditionally** rather than being activated uniformly at every candidate;
2. **immediate maximum exposure is not supported** by the forward-return evidence, making timing and progressive accumulation explicit research questions;
3. **bear→bull and bull→bear transitions must be allowed different sizing and entry policies**;
4. **failed-transition invalidation must be explicitly modelled**, because high-confidence failures can generate substantial adverse returns;
5. **anticipatory exposure must ultimately be integrated with conventional trend exposure and handed off at confirmation** rather than evaluated as an isolated event trade.

Book 01 prototype return series remain suitable for this research-stage event study but should not yet be interpreted as institutional-quality total-return histories for every asset class. The `PNL_PROTOTYPE` designation primarily identifies the preferred Book 01 research return series and does not by itself establish complete treatment of futures roll and collateral returns, FX carry, dividends or implementation costs. These remain requirements of the final portfolio engine.

### Conclusion

Book 06 establishes that the frozen Book 04 transition probability contains **economically meaningful forward-return information**, particularly from approximately the 21-observation horizon onward and most clearly for bear→bull transitions and equity indices.

The result also clarifies the nature of that information. The model appears to identify **developing trend transitions rather than exact turning points**: essentially no top-quintile economic advantage is observed after only 5 observations, while progressively stronger separation emerges over subsequent weeks and months.

Book 06 therefore passes the economic-validation gate for continued development, but **does not validate immediate entry, a symmetric standalone transition strategy, or indiscriminate high-probability trading**.

The next research stage should construct and test the actual transition state machine:

\[
\text{Established Trend}
\rightarrow
\text{Transition Candidate}
\rightarrow
P(\text{Genuine Transition})
\rightarrow
\text{Progressive Anticipatory Exposure}
\rightarrow
\begin{cases}
\text{Invalidation} \\
\text{Conventional Confirmation}
\end{cases}
\rightarrow
\text{Conventional Trend Hand-off}.
\]

Book 07 should therefore determine whether this event-level economic information can improve an implemented conventional CTA once timing, accumulation, invalidation, overlapping signals, direction asymmetry, volatility scaling and portfolio capital constraints are imposed.

**Status: FROZEN as the V2 event-level economic validation. Transition probability PASSES the economic-information test; immediate entry and naive symmetric implementation are REJECTED.**